In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
Path.cwd()

In [ ]:
root_dir = Path.cwd().parent
temp_dir = root_dir / ".temp"
assert temp_dir.exists()

temp_dir

In [ ]:
dataset_path = temp_dir / "vehicles.csv"
dataset_path

In [ ]:
images_path = Path.cwd() / ".." / "charged-ieee" / "images"
images_path

Globals

In [ ]:
RNG = 99

# Load

In [ ]:
df = pd.read_csv(dataset_path)
df_og = df.copy()
df_og

# General

In [ ]:
len(df)

In [ ]:
df.describe()

In [ ]:
print("Mean Price:", df["price"].mean())
print("Median Price:", df["price"].median())

In [ ]:
df.columns

In [ ]:
cols = df.columns.tolist()
cols_drop = [
    "id",
    "url",
    "region_url",
    "image_url",
    "description",
    "county",  # (all missing)
    "VIN",
    "lat",
    "long",
    "posting_date",
    "model",  # (not determined, too many)
    "size",  # (72% missing)
]

col_label = "price"  # (complete -> required)
cols_features_num = [
    "year",  # (1% missing -> drop)
    "odometer",  # (1% missing -> drop)
]
cols_features_cat = [
    "region",  # target encode (complete -> required)
    "state",  # target encode (complete -> required)
    #
    "manufacturer",  # target encode (4% missing -> mark)
    #
    "condition",  # onehot encode (41% missing)
    "cylinders",  # onehot encode (42% missing)
    "fuel",  # onehot encode (1% missing)
    "title_status",  # onehot encode (2% missing)
    "transmission",  # onehot encode (1% missing)
    "drive",  # onehot encode (31% missing)
    "type",  # onehot encode (22% missing)
    "paint_color",  # onehot encode (31% missing)
]

assert set(cols) == (
    set(cols_drop) | set(cols_features_num) | set(cols_features_cat) | {col_label}
)

# Plots

#### Settings

In [ ]:
fm.fontManager.addfont(Path.cwd() / ".." / "resources" / "LibertinusSerif-Regular.ttf")

plt.rc("font", size=9)
plt.rcParams["figure.dpi"] = 500
# plt.style.use("dark_background")
plt.rcParams["font.family"] = "Libertinus Serif"

### Plots

In [ ]:
df = df_og.copy()

Price distribution

In [ ]:
plt.figure(figsize=(3.5, 2.2))
plt.hist(df[df["price"] < 80000]["price"], bins=150)
plt.ylim(0, 20000)
plt.xlabel("price")
plt.ylabel("count")

#plt.savefig(images_path / "price_distribution.png", bbox_inches="tight", pad_inches=0)
plt.show()

KDE year and price

In [ ]:
df["year"].min(), df["year"].max()

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.2))

filtered = df[(df["price"] < 80000) & (df["year"] >= 1980)]

sns.kdeplot(
    data=filtered,
    x="year",
    y="price",
    ax=ax,
    fill=True,
    cmap="viridis",
    levels=10,
    thresh=0.02,
    bw_adjust=0.75,
)

kde = sns.kdeplot(
    data=filtered,
    x="year",
    bw_adjust=1.0,
    ax=ax,
    color="red",
    linewidth=1.5,
    alpha=0.65,
)

line = kde.lines[-1]
x, y = line.get_data()

y_scaled = (y / y.max()) * 80000
y_scaled *= 0.8

line.set_data(x, y_scaled)

ax.set_xlim(1980, 2023)
ax.set_ylim(0, 80000)

fig.tight_layout()
#plt.savefig(images_path / "year_price_heatmap.png", bbox_inches="tight", pad_inches=0)
plt.show()

Odometer distribution

In [ ]:
plt.figure(figsize=(3.5, 2.2))
plt.hist(df[df["odometer"] <= 300000]["odometer"], bins=150)
plt.ylim(0, 10000)
plt.xlabel("odometer")
plt.ylabel("count")

#plt.savefig(images_path / "odometer_distribution.png", bbox_inches="tight", pad_inches=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.2))

filtered = df[(df["price"] < 80000) & (df["odometer"] <= 300000)]

sns.kdeplot(
    data=filtered,
    x="odometer",
    y="price",
    ax=ax,
    fill=True,
    cmap="viridis",
    levels=10,
    thresh=0.02,
    bw_adjust=0.75,
)

kde = sns.kdeplot(
    data=filtered,
    x="odometer",
    bw_adjust=1.0,
    ax=ax,
    color="red",
    linewidth=1.5,
    alpha=0.65,
)

line = kde.lines[-1]
x, y = line.get_data()

y_scaled = (y / y.max()) * 80000
y_scaled *= 0.8

line.set_data(x, y_scaled)

ax.set_xlim(0, 300000)
ax.set_ylim(0, 80000)

fig.tight_layout()
#plt.savefig(images_path / "odometer_price_heatmap.png", bbox_inches="tight", pad_inches=0)
plt.show()

Analyse Data with boundaries applied

In [ ]:
dz = df_og.copy()[[col_label] + cols_features_num + cols_features_cat]
dz = dz[(dz["price"] >= 500) & (dz["price"] <= 80000)]
dz = dz[dz["year"] >= 1980]
dz = dz[dz["odometer"] <= 400000]
dz = dz[dz["year"].notna() & dz["odometer"].notna()]

# Preprocess

In [ ]:
df = df_og.copy()[[col_label] + cols_features_num + cols_features_cat]

### Remove Boundaries

Remove all entries where price is lower than 500 or higher than 80000

In [ ]:
df = df[(df["price"] >= 500) & (df["price"] <= 80000)]
len(df)

Remove all entries where year is lower than 1980

In [ ]:
df = df[df["year"] >= 1980]
len(df)

Remove all entries where odometer is over 400000

In [ ]:
df = df[df["odometer"] <= 400000]
len(df)

### Drop Missing

In [ ]:
df = df[df["year"].notna() & df["odometer"].notna()]
len(df)

### Require Complete

In [ ]:
cols_complete = [col_label, "region", "state"]

for c in cols_complete:
    assert c in df.columns, f"Missing required column: {c}"
    assert df[c].notna().all(), f"Column '{c}' contains missing values"

### Mark Missing

In [ ]:
def mark_missing(df, col):
    df[f"{col}_missing"] = df[col].isna().astype(int)


for col in ["manufacturer"]:
    mark_missing(df, col)

### Train Test Split

In [ ]:
cols_features = [c for c in df.columns if c != col_label]

X_train, X_test, y_train, y_test = train_test_split(
    df[cols_features],
    df[col_label],
    test_size=0.25,
    random_state=RNG,
)

### Result

In [ ]:
X_train

In [ ]:
X_train.shape, X_test.shape

### Save

In [ ]:
pd.concat([X_train, y_train], axis=1).to_csv(
    temp_dir / "vehicles_train.csv", index=False
)
pd.concat([X_test, y_test], axis=1).to_csv(temp_dir / "vehicles_test.csv", index=False)